1. Reload data

In [16]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

2. Clean TotalCharges

Why?

The model cannot properly treat a numeric variable as text.

We convert invalid/blank values to NaN, then handle them systematically.

In [17]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(df["TotalCharges"].dtype)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

float64
Missing TotalCharges: 11


3. Remove customerID

Why?

customerID identifies the customer but does not represent a useful customer behavior feature.

Keeping it could cause the model to learn meaningless identifiers.

In [18]:
df = df.drop(columns=["customerID"])

4. Convert target

In [19]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

5. Separate X and y

Why?

Machine learning follows:

X → Features
y → Target

We want the model to learn:

Customer characteristics → Probability of churn



In [20]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

6. Train/test split

Why stratify=y?

Because churn is imbalanced.

We want the train and test datasets to maintain approximately the same churn distribution.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (5634, 19)
Testing: (1409, 19)


7. Identify column types

In [22]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\lalil\AppData\Local\Temp\ipykernel_37472\1755318793.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


8. Build preprocessing pipeline

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

9. Save processed datasets

In [24]:
processed_path = PROJECT_ROOT / "data" / "processed"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

X_train.to_csv(
    processed_path / "X_train.csv",
    index=False
)

X_test.to_csv(
    processed_path / "X_test.csv",
    index=False
)

y_train.to_csv(
    processed_path / "y_train.csv",
    index=False
)

y_test.to_csv(
    processed_path / "y_test.csv",
    index=False
)